<a href="https://colab.research.google.com/github/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project/blob/main/261%20-%20MHCBVUD%20-%20L01%20-%20Nh%C3%B3m%2010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nhóm 10 – Dự đoán tái nhập viện ở bệnh nhân đái tháo đường (Diabetes 130-US)
**Notebook tổng hợp:** chạy **script của từng thành viên, lấy trực tiếp từ nhánh của người đó**, theo thứ tự từ trên xuống. Các bước nối với nhau bằng file trong `output/`: output của bước trước là input của bước sau. `Runtime → Run all` là chạy cả pipeline.

| Bước | Task | Người | Script (nhánh) | Đọc | Ghi (`output/`) |
|---|---|---|---|---|---|
| 01 | Bỏ thuộc tính định danh | Dũng | `src/01_explore_and_drop_ids.py` (Dung) | `data/diabetic_data.csv` | `step1.csv`, `step1_patient_nbr.csv` |
| 02 | Thống kê khuyết, bỏ cột khuyết > 50% | Sơn | `src/02_missing_stats_and_filter.py` (Son) | `step1.csv` | `step2.csv`, `missing_rate.csv` |
| 03 | Bỏ dòng khuyết < 2%, tạo nhãn, chia 80/20 | Chiến | `src/03_data_preprocessing_splitting.py` (Chien) | `step2.csv`, `step1_patient_nbr.csv` | `train.csv`, `test.csv` |
| 04 | One-Hot (fit trên train) | Dũng | `src/04_one_hot_encoding.py` (Dung) | `train.csv`, `test.csv` | `X_train_onehot.npz`, `X_test_onehot.npz`, `onehot_columns.txt`, `y_train.csv`, `y_test.csv` |
| 05 | PCA 60 thành phần | Sơn | `src/05_pca.py` (Son) | `X_*_onehot.npz`, `onehot_columns.txt` | `X_train_pca.npy`, `X_test_pca.npy`, `pca_eigenvalues.npy`, `pca_components.npy`, `pca_feature_names.txt` |
| 06 | Chuẩn hóa sau PCA, scree plot, top 5 vector riêng | Chiến | `src/06_feature_scaling_pca_analysis.py` (Chien) | `X_*_pca.npy`, `pca_*` | `X_train_pca_scaled.npy`, `X_test_pca_scaled.npy`, `eigenvalues.png` |
| 07a | k-NN tự cài, PCA chưa vs đã chuẩn hóa | Dũng | `src/07a_knn.py` (Dung) | `X_*_pca*.npy`, `y_*.csv` | `knn_results.csv`, `knn_k_curve.png` |
| 07b | Softmax Regression tự cài, chưa vs đã chuẩn hóa | Dũng | `src/07b_softmax_regression.py` (Dung) | `X_*_pca*.npy`, `y_*.csv` | `softmax_results.csv`, `softmax_loss.png` |
| 08a | Decision Tree, tinh chỉnh max_depth | Sơn | `src/08a_decision_tree.py` (Son) | `X_*_pca.npy`, `y_*.csv` | `dt_results.csv`, `dt_depth_curve.png`, `pred_multi_dt.npy`, `proba_{val,test}_dt.npy` |
| 08b | Random Forest, tinh chỉnh max_depth, n_estimators | Sơn | `src/08b_random_forest.py` (Son) | `X_*_pca.npy`, `y_*.csv` | `rf_results.csv`, `rf_depth_curve.png`, `pred_multi_rf.npy`, `proba_{val,test}_rf.npy` |
| 09a | MLP 120-60-20, ReLU | Chiến | `src/09a_mlp.py` (Chien) | `X_*_pca_scaled.npy`, `y_*.csv` | `mlp_results.csv`, `mlp_alpha_curve.png`, `mlp_loss.png`, `pred_multi_mlp.npy`, `proba_{val,test}_mlp.npy` |
| 09b | Ensemble trọng số AUC (DT, RF, MLP, LR) | Chiến | `src/09b_ensemble.py` (Chien) | `proba_{val,test}_*.npy` | `ensemble_weights.csv`, `ensemble_results.csv`, `proba_test_ensemble.npy` |
| 10 | Test Accuracy multi-class, ma trận nhầm lẫn | Dũng | `src/10_evaluation.py` (Dung) | `pred_*.npy`, `y_test.csv` | `accuracy_summary.csv`, `confusion_*.csv`, `confusion_matrices.png` |
| 11 | Test Accuracy nhị phân, ROC/AUC | Sơn | `src/11_binary_evaluation.py` (Son) | `proba_test_*.npy`, `y_test.csv` | `binary_summary.csv`, `roc_curves.png`, `confusion_binary.png` |
| 12 | Class Imbalance; RF trên One-Hot chưa PCA | Chiến | `src/12_imbalance_and_rf_raw.py` (Chien) | `y_*.csv`, `X_*_onehot.npz`, `rf_results.csv`, `proba_*_rf.npy` | `imbalance_*.csv`, `rf_raw_*.csv`, `rf_raw_importance.png` |

**Quy ước:** mọi script chạy từ thư mục gốc dự án, đọc/ghi đúng tên file trong bảng; không đổi tên khi chưa báo cả nhóm. Các bước học từ dữ liệu chỉ fit trên train. `random_state = 42`. Muốn sửa một bước thì sửa script trên nhánh của mình rồi push, chạy lại notebook là cập nhật.

## 0. Cài đặt
Clone cả bốn nhánh (`main`, `Dung`, `Son`, `Chien`) vào `/content/branches/`. Thư mục làm việc là `branches/main` (có dữ liệu gốc `data/diabetic_data.csv`); script của từng người được gọi qua `../Dung/src/...`, `../Son/src/...`, `../Chien/src/...`, nên tất cả cùng đọc/ghi một thư mục `output/`.

Phiên bản thư viện được ghim (`scikit-learn==1.6.1`, `pandas==2.2.3`, `numpy==2.4.6`): cách chia train/test ở bước 03 phụ thuộc phiên bản scikit-learn, khác phiên bản sẽ ra tập train/test khác (79.726 / 20.518 dòng với bản 1.6.1). Kiểm tra dòng in phiên bản ở cuối cell bên dưới trước khi chạy tiếp.


In [ ]:
REPO = "https://github.com/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project.git"
%cd /content
!rm -rf /content/branches
for b in ["main", "Dung", "Son", "Chien"]:
    !git clone -q -b {b} {REPO} /content/branches/{b}
    !git -C /content/branches/{b} log -1 --format="{b}: %h %an %ad %s" --date=short
%cd /content/branches/main
!pip -q install -r ../Dung/requirements.txt -r ../Son/requirements.txt -r ../Chien/requirements.txt
!pip -q install scikit-learn==1.6.1 pandas==2.2.3 numpy==2.4.6
!python -c "import sklearn, pandas, numpy; print('sklearn', sklearn.__version__, '| pandas', pandas.__version__, '| numpy', numpy.__version__)"

## Giai đoạn 1: Khảo sát và xử lý dữ liệu khuyết

### Bước 01: Bỏ thuộc tính định danh *(Dũng)*

In [ ]:
!python ../Dung/src/01_explore_and_drop_ids.py

### Bước 02: Thống kê khuyết, bỏ cột khuyết > 50% *(Sơn)*

In [ ]:
!python ../Son/src/02_missing_stats_and_filter.py

### Bước 03: Bỏ dòng khuyết < 2%, tạo nhãn, chia train/test 80/20 *(Chiến)*

In [ ]:
!python ../Chien/src/03_data_preprocessing_splitting.py

## Giai đoạn 2: One-Hot, PCA, chuẩn hóa

### Bước 04: One-Hot Encoding *(Dũng)*

In [ ]:
!python ../Dung/src/04_one_hot_encoding.py

### Bước 05: PCA 60 thành phần trên ma trận tương quan *(Sơn)*

In [ ]:
!python ../Son/src/05_pca.py

### Bước 06: Chuẩn hóa sau PCA, biểu đồ Eigenvalues, top 5 vector riêng *(Chiến)*

In [ ]:
!python ../Chien/src/06_feature_scaling_pca_analysis.py

### Kết quả Giai đoạn 2

In [ ]:
from IPython.display import Image, display
!ls -la output
display(Image("output/eigenvalues.png"))

## Giai đoạn 3: Huấn luyện mô hình

### Bước 07a: Lập trình và tối ưu k-NN (trên dữ liệu chưa chuẩn hoá và đã chuẩn hoá) *(Dũng)*

In [ ]:
!python ../Dung/src/07a_knn.py
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("output/knn_results.csv")); display(Image("output/knn_k_curve.png"))

### Bước 07b: Softmax Regression, chưa chuẩn hóa vs đã chuẩn hóa *(Dũng)*


In [ ]:
!python ../Dung/src/07b_softmax_regression.py
display(pd.read_csv("output/softmax_results.csv")); display(Image("output/softmax_loss.png"))

### Bước 08a: Decision Tree, tinh chỉnh max_depth chống overfitting *(Sơn)*

In [ ]:
import pandas as pd
from IPython.display import Image, display
!python -u ../Son/src/08a_decision_tree.py
display(pd.read_csv("output/dt_results.csv"))
display(Image("output/dt_depth_curve.png"))

### Bước 08b: Random Forest, tinh chỉnh max_depth và n_estimators *(Sơn)*

In [ ]:
!python -u ../Son/src/08b_random_forest.py
display(pd.read_csv("output/rf_results.csv"))
display(Image("output/rf_depth_curve.png"))

### Bước 09a: Deep Neural Network / MLP (120-60-20, ReLU) *(Chiến)*

In [ ]:
!python -u ../Chien/src/09a_mlp.py
display(pd.read_csv("output/mlp_results.csv"))
display(Image("output/mlp_alpha_curve.png"))
display(Image("output/mlp_loss.png"))

### Bước 09b: Aggregate Ensemble theo trọng số AUC-ROC *(Chiến)*

In [ ]:
!python -u ../Chien/src/09b_ensemble.py
display(pd.read_csv("output/ensemble_weights.csv"))
display(pd.read_csv("output/ensemble_results.csv"))

## Giai đoạn 4: Đánh giá tổng hợp

### Bước 10: Test Accuracy multi-class, ma trận nhầm lẫn *(Dũng)*


In [ ]:
import pandas as pd
from IPython.display import Image, display
!python ../Dung/src/10_evaluation.py
display(pd.read_csv("output/accuracy_summary.csv"))
display(Image("output/confusion_matrices.png"))

### Bước 11: Test Accuracy bài toán nhị phân, đường cong ROC/AUC *(Sơn)*

In [ ]:
!python -u ../Son/src/11_binary_evaluation.py
display(pd.read_csv("output/binary_summary.csv"))
display(Image("output/roc_curves.png"))
display(Image("output/confusion_binary.png"))

### Bước 12: Ảnh hưởng của Class Imbalance; Random Forest trên One-Hot chưa PCA *(Chiến)*

In [ ]:
!python -u ../Chien/src/12_imbalance_and_rf_raw.py
for f in ["imbalance_distribution", "imbalance_rf_multi", "imbalance_threshold", "rf_raw_results"]:
    print(f); display(pd.read_csv(f"output/{f}.csv"))
display(pd.read_csv("output/rf_raw_importance_orig.csv").head(15))
display(Image("output/rf_raw_importance.png"))